# Deep Learning Project: Multiclass Classification of Sequential Process Data

## Group 1

**Group ID:** 1

- Manotas Ramos, Rodolfo A. — 5054703
- Schnerr, Georg — 5188132
- Schötzau, Jannes — xxxxxxx

## 0. Project Objective

This project develops a deep-learning model that classifies machine condition from sequential acceleration measurements. The supplied configuration uses the `Acc_X` and `Acc_Y` signals and distinguishes the classes `Gut`, `Mangelschmierung`, and `Pitting_V1`.

The required final test accuracy is at least 55%. The workflow uses a baseline 1D convolutional neural network (CNN), Optuna hyperparameter optimization, an optimized CNN, and a final evaluation on data that remains unseen until the end.

In [ ]:
# Classification configuration supplied for Group 1; do not modify.
selected_axes = ['Acc_X', 'Acc_Y']
selected_classes = ['Gut', 'Mangelschmierung', 'Pitting_V1']

## 1. Setup and Configuration

In [ ]:
import copy
import random
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import Markdown, display
from matplotlib.lines import Line2D
from optuna.samplers import TPESampler
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

A fixed seed is declared before any model result is observed. It is not selected for a high score. Using the same data split, weight initialization, and Optuna sampling sequence makes the baseline and optimized model directly comparable and allows the notebook to be reproduced. A single seed does not prove robustness across all possible initializations, but it provides a transparent and fair experiment for this project.

In [ ]:
SEED = 42


def set_seed(seed=SEED):
    """Seed the random-number generators used by this notebook."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()
print(f"Reproducibility seed: {SEED}")

## 2. Data Loading

In [ ]:
# The dataset is loaded locally from the repository root.
# This avoids user-specific and operating-system-specific paths.
data_path = Path("Datensatz.csv")
if not data_path.is_file():
    raise FileNotFoundError(
        "Datensatz.csv was not found. Place it in the repository root."
    )

# Load only a small preview here; the protected preprocessing function loads the full dataset.
data_preview = pd.read_csv(data_path, nrows=5)
display(data_preview)

## 3. Data Preprocessing — Protected

The following two functions are supplied by the assignment and must not be changed. The only local adaptation is `file_path = data_path`, which replaces the original machine-specific Windows path without changing the preprocessing logic.

In [ ]:
####################################################################################################################################################################################################################################################### DIESEN BLOCK NICHT ÄNDERN #########################################################################################
#######################################################################################################################################################################################

def to_categorical(y, num_classes):
    """ 1-hot encodes a tensor """
    return np.eye(num_classes, dtype='uint8')[y]

def data_preprocessing(selected_axes=['Acc_X', 'Acc_Y', 'Acc_Z'], selected_classes=['Gut', 'Mangelschmierung', 'Pitting_V1', 'Pitting_V2', 'Pitting_V3', 'Pitting_V4'], x_len=2500):  # Select which Acc-Data should be included axis: ['Acc_X', 'Acc_Y', 'Acc_Z']
    file_path = data_path
    df = pd.read_csv(file_path)
    df = df.drop(df.columns[0:2], axis=1)
    df = df[df['Status'].isin(selected_classes)]

    maxlen = df['run_ID'].value_counts().max()

    if len(selected_classes) < 6:
        label_arr_old = df['label'].values
        label_arr_new = np.zeros(shape=label_arr_old.shape)
        label_arr = np.vstack((label_arr_old, label_arr_new)).T
        i = int(0)
        for idx in np.unique(label_arr_old):
            label_arr[np.where(label_arr[:, 0] == idx), 1] = i
            i += 1
        label_arr = label_arr.astype(int)
        df['label'] = df['label'].map(dict(label_arr))

    n_classes = len(df['label'].value_counts())
    Load = df[['label', 'run_ID']]
    drop_cols = ['label', 'Status', ]
    drop_cols = list(set(drop_cols) & set(df.columns))  # Check if cols are part of df and select only those
    df = df.drop(columns=drop_cols).reset_index(drop=True)
    input_column_names = selected_axes
    x_columns = input_column_names + ['run_ID']
    g = df.groupby('run_ID').cumcount()
    X = (df[x_columns].set_index(['run_ID', g])
         .unstack(fill_value=0)
         .stack().groupby(level=0)
         .apply(lambda x: x.values)
         .to_numpy())
    X = np.array([i for i in X])
    if x_len is not None:
        X = X[:, 0:x_len, :]
        maxlen = x_len
    # del df  <- in case of memory issues
    g = Load.groupby('run_ID').cumcount()
    y = (Load.set_index(['run_ID', g])
         .unstack(fill_value=0)
         .stack().groupby(level=0)
         .apply(lambda x: x.values[0])
         .to_numpy().astype("int32"))
    y = to_categorical(y, n_classes)
    return X, y, maxlen

## 4. Dataset Preparation

### 4.1 Dataset Class

In [ ]:
class DataSet(Dataset):
    """Wrap NumPy arrays as a PyTorch dataset."""

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]).float(), torch.from_numpy(self.y[idx]).float()

### 4.2 Preprocessing and Stratified Split

The supplied preprocessing code triggers a NumPy deprecation warning but remains operational with the pinned package versions. The warning is suppressed around this call so that the protected implementation can remain unchanged.

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=DeprecationWarning)
    X, y, maxlen = data_preprocessing(
        selected_axes=selected_axes,
        selected_classes=selected_classes,
    )

labels = np.argmax(y, axis=1)

# Keep the test set isolated and preserve class proportions in every split.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    shuffle=True,
    stratify=labels,
    random_state=SEED,
)

train_labels_before_validation_split = np.argmax(y_train, axis=1)
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.33,
    shuffle=True,
    stratify=train_labels_before_validation_split,
    random_state=SEED,
)

sequence_length = X.shape[1]
n_features = X.shape[2]
n_classes = len(selected_classes)

print(f"Full dataset: X={X.shape}, y={y.shape}")
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Validation: X={X_val.shape}, y={y_val.shape}")
print(f"Test: X={X_test.shape}, y={y_test.shape}")

### 4.3 Normalization and DataLoaders

The scaler is fitted on the training set only. Validation and test data are transformed with the same fitted scaler, which prevents information leakage.

In [ ]:
scaler = StandardScaler().fit(X_train.reshape(-1, n_features))


def scale_sequences(values):
    """Apply the training-set scaler while preserving sequence dimensions."""
    return scaler.transform(values.reshape(-1, n_features)).reshape(values.shape)


X_train = scale_sequences(X_train)
X_val = scale_sequences(X_val)
X_test = scale_sequences(X_test)

train_dataset = DataSet(X_train, y_train)
val_dataset = DataSet(X_val, y_val)
test_dataset = DataSet(X_test, y_test)

BATCH_SIZE = 64


def make_loader(dataset, shuffle=False, seed=SEED, batch_size=BATCH_SIZE):
    """Create a reproducible DataLoader."""
    generator = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator,
    )


val_loader = make_loader(val_dataset)
test_loader = make_loader(test_dataset)

### 4.4 Dataset Overview

The first visualization checks whether the three machine-condition classes remain similarly represented after the stratified split.

In [ ]:
split_class_counts = pd.DataFrame(
    {
        "Train": np.bincount(np.argmax(y_train, axis=1), minlength=n_classes),
        "Validation": np.bincount(np.argmax(y_val, axis=1), minlength=n_classes),
        "Test": np.bincount(np.argmax(y_test, axis=1), minlength=n_classes),
    },
    index=selected_classes,
)

display(split_class_counts)

ax = split_class_counts.plot(
    kind="bar",
    figsize=(10, 5),
    color=["#4C78A8", "#F58518", "#54A24B"],
)
ax.set_title("Class Distribution Across Dataset Splits")
ax.set_xlabel("Machine condition")
ax.set_ylabel("Number of runs")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Baseline 1D-CNN

### 5.1 Architecture and Rationale

A 1D-CNN is appropriate because the input consists of ordered acceleration sequences. Convolutional filters can identify local temporal patterns such as peaks, oscillations, and changes in vibration behavior. Two input channels represent `Acc_X` and `Acc_Y`; three output logits represent the required machine-condition classes.

The output layer does not apply Softmax because `CrossEntropyLoss` combines the required log-softmax operation with the classification loss.

In [ ]:
class CNN1D(nn.Module):
    """Two-block 1D-CNN used for both baseline and optimized configurations."""

    def __init__(self, fc_size=64):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, 16, kernel_size=5, padding=2)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=5)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=5)
        self.flatten = nn.Flatten()
        pooled_length = sequence_length // 25
        self.fc1 = nn.Linear(32 * pooled_length, fc_size)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(fc_size, n_classes)

    def forward(self, x):
        # Convert (batch, sequence, features) to (batch, features, sequence).
        x = x.permute(0, 2, 1)
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        return self.fc2(x)


def count_trainable_parameters(model):
    """Return the number of trainable model parameters."""
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

### 5.2 Training and Evaluation Utilities

In [ ]:
def evaluate_model(model, loader, criterion):
    """Evaluate a model and return loss, accuracy, labels, and predictions."""
    model.eval()
    total_loss = 0.0
    total_samples = 0
    all_true = []
    all_pred = []

    with torch.no_grad():
        for batch_X, batch_y in loader:
            logits = model(batch_X.float())
            true_classes = torch.argmax(batch_y, dim=1).long()
            loss = criterion(logits, true_classes)
            predicted_classes = torch.argmax(logits, dim=1)

            batch_size = true_classes.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size
            all_true.extend(true_classes.cpu().numpy())
            all_pred.extend(predicted_classes.cpu().numpy())

    all_true = np.asarray(all_true)
    all_pred = np.asarray(all_pred)
    return {
        "loss": total_loss / total_samples,
        "accuracy": accuracy_score(all_true, all_pred),
        "y_true": all_true,
        "y_pred": all_pred,
    }


def train_model(model, learning_rate, epochs, seed=SEED, verbose=True):
    """Train a model and restore the checkpoint with the best validation accuracy."""
    set_seed(seed)
    train_loader = make_loader(train_dataset, shuffle=True, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": [],
    }
    best_state = None
    best_val_accuracy = -np.inf
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        samples_seen = 0

        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model(batch_X.float())
            target_indices = torch.argmax(batch_y, dim=1).long()
            loss = criterion(logits, target_indices)
            loss.backward()
            optimizer.step()

            batch_size = target_indices.size(0)
            running_loss += loss.item() * batch_size
            samples_seen += batch_size

        val_results = evaluate_model(model, val_loader, criterion)
        history["train_loss"].append(running_loss / samples_seen)
        history["val_loss"].append(val_results["loss"])
        history["val_accuracy"].append(val_results["accuracy"])

        if val_results["accuracy"] > best_val_accuracy:
            best_val_accuracy = val_results["accuracy"]
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())

        if verbose and ((epoch + 1) == 1 or (epoch + 1) % 5 == 0 or (epoch + 1) == epochs):
            print(
                f"Epoch {epoch + 1:02d}/{epochs} | "
                f"Train Loss: {history['train_loss'][-1]:.4f} | "
                f"Validation Accuracy: {val_results['accuracy'] * 100:.2f}%"
            )

    model.load_state_dict(best_state)
    return model, history, best_epoch

### 5.3 Baseline Training and Validation

The baseline and optimized model use the same 30-epoch budget and the same seed. This makes the later comparison fair: improvements can be attributed to the optimized hyperparameters rather than to a longer training run or a different random split.

In [ ]:
BASELINE_FC_SIZE = 64
BASELINE_LEARNING_RATE = 1e-3
FINAL_TRAINING_EPOCHS = 30

set_seed()
baseline_model = CNN1D(fc_size=BASELINE_FC_SIZE)
baseline_model, baseline_history, baseline_best_epoch = train_model(
    baseline_model,
    learning_rate=BASELINE_LEARNING_RATE,
    epochs=FINAL_TRAINING_EPOCHS,
)
baseline_criterion = nn.CrossEntropyLoss()
baseline_val_results = evaluate_model(baseline_model, val_loader, baseline_criterion)

print(f"Best baseline epoch: {baseline_best_epoch}")
print(f"Baseline validation accuracy: {baseline_val_results['accuracy'] * 100:.2f}%")

## 6. Hyperparameter Optimization with Optuna

### 6.1 Search Space and Objective

Optuna tunes the Adam learning rate and the size of the fully connected layer. Every trial uses the same split, seed, and training budget so that its validation score is comparable. The test set is not accessed during optimization.

In [ ]:
OPTUNA_TRIALS = 15
OPTUNA_EPOCHS = 20


def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    fc_size = trial.suggest_categorical("fc_size", [32, 64, 128])

    set_seed()
    trial_model = CNN1D(fc_size=fc_size)
    trial_model, trial_history, _ = train_model(
        trial_model,
        learning_rate=learning_rate,
        epochs=OPTUNA_EPOCHS,
        verbose=False,
    )
    return max(trial_history["val_accuracy"])


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=OPTUNA_TRIALS)

best_learning_rate = study.best_params["learning_rate"]
best_fc_size = study.best_params["fc_size"]

print(f"Best Optuna validation accuracy: {study.best_value * 100:.2f}%")
print(f"Best learning rate: {best_learning_rate:.6f}")
print(f"Best fully connected size: {best_fc_size}")

## 7. Optimized 1D-CNN

### 7.1 Champion Training and Validation

The optimized model is trained from scratch with Optuna's best hyperparameters. As with the baseline, the state from the epoch with the highest validation accuracy is restored before comparison.

In [ ]:
set_seed()
champion_model = CNN1D(fc_size=best_fc_size)
champion_model, champion_history, champion_best_epoch = train_model(
    champion_model,
    learning_rate=best_learning_rate,
    epochs=FINAL_TRAINING_EPOCHS,
)
champion_criterion = nn.CrossEntropyLoss()
champion_val_results = evaluate_model(champion_model, val_loader, champion_criterion)

print(f"Best champion epoch: {champion_best_epoch}")
print(f"Champion validation accuracy: {champion_val_results['accuracy'] * 100:.2f}%")

### 7.2 Learning Curves

Training and validation curves reveal convergence behavior and help identify underfitting or overfitting. The dashed vertical lines mark the best validation epoch restored for each model.

In [ ]:
baseline_epochs = np.arange(1, len(baseline_history["train_loss"]) + 1)
champion_epochs = np.arange(1, len(champion_history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

axes[0].plot(baseline_epochs, baseline_history["train_loss"], label="Baseline train loss", color="#4C78A8")
axes[0].plot(baseline_epochs, baseline_history["val_loss"], label="Baseline validation loss", color="#4C78A8", linestyle="--")
axes[0].plot(champion_epochs, champion_history["train_loss"], label="Optimized train loss", color="#F58518")
axes[0].plot(champion_epochs, champion_history["val_loss"], label="Optimized validation loss", color="#F58518", linestyle="--")
axes[0].set_title("Loss Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].plot(baseline_epochs, np.array(baseline_history["val_accuracy"]) * 100, label="Baseline", color="#4C78A8")
axes[1].plot(champion_epochs, np.array(champion_history["val_accuracy"]) * 100, label="Optimized", color="#F58518")
axes[1].axvline(baseline_best_epoch, color="#4C78A8", linestyle=":", alpha=0.8)
axes[1].axvline(champion_best_epoch, color="#F58518", linestyle=":", alpha=0.8)
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].grid(alpha=0.25)
axes[1].legend()

fig.subplots_adjust(left=0.07, right=0.98, bottom=0.12, top=0.90, wspace=0.34)
fig.canvas.draw()
separator_x = (
    axes[0].get_position().x1 + axes[1].get_position().x0
) / 2
fig.add_artist(
    Line2D(
        [separator_x, separator_x],
        [0.06, 0.94],
        transform=fig.transFigure,
        color="#D9D9D9",
        linewidth=1,
        zorder=0,
    )
)
plt.show()

### 7.3 Optuna Optimization History

The optimization history shows both the progression of validation accuracy across trials and the relationship between learning rate, fully connected size, and performance.

In [ ]:
trials_df = study.trials_dataframe()
completed_trials = trials_df[trials_df["state"] == "COMPLETE"].copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

axes[0].plot(completed_trials["number"], completed_trials["value"] * 100, marker="o", color="#4C78A8")
axes[0].scatter(
    study.best_trial.number,
    study.best_value * 100,
    color="#E45756",
    s=100,
    zorder=3,
    label="Best trial",
)
axes[0].set_title("Optuna Validation Accuracy by Trial")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("Validation accuracy (%)")
axes[0].grid(alpha=0.25)
axes[0].legend()

scatter = axes[1].scatter(
    completed_trials["params_learning_rate"],
    completed_trials["value"] * 100,
    c=completed_trials["params_fc_size"],
    cmap="viridis",
    s=80,
)
axes[1].set_xscale("log")
axes[1].set_title("Hyperparameters and Validation Accuracy")
axes[1].set_xlabel("Learning rate")
axes[1].set_ylabel("Validation accuracy (%)")
axes[1].grid(alpha=0.25)
colorbar = fig.colorbar(scatter, ax=axes[1])
colorbar.set_label("Fully connected size")

fig.subplots_adjust(left=0.07, right=0.94, bottom=0.12, top=0.90, wspace=0.38)
fig.canvas.draw()
separator_x = (
    axes[0].get_position().x1 + axes[1].get_position().x0
) / 2
fig.add_artist(
    Line2D(
        [separator_x, separator_x],
        [0.06, 0.94],
        transform=fig.transFigure,
        color="#D9D9D9",
        linewidth=1,
        zorder=0,
    )
)
plt.show()

## 8. Model Comparison

Both configurations are compared on the same validation set. Test accuracy is intentionally excluded from this table because the test set remains untouched until the final section.

In [ ]:
def summarize_validation_results(name, model, learning_rate, fc_size, best_epoch, results):
    precision, recall, f1, _ = precision_recall_fscore_support(
        results["y_true"],
        results["y_pred"],
        average="macro",
        zero_division=0,
    )
    return {
        "Model": name,
        "Learning rate": learning_rate,
        "FC size": fc_size,
        "Best epoch": best_epoch,
        "Parameters": count_trainable_parameters(model),
        "Validation accuracy": results["accuracy"],
        "Macro precision": precision,
        "Macro recall": recall,
        "Macro F1": f1,
    }


comparison_df = pd.DataFrame(
    [
        summarize_validation_results(
            "Baseline CNN",
            baseline_model,
            BASELINE_LEARNING_RATE,
            BASELINE_FC_SIZE,
            baseline_best_epoch,
            baseline_val_results,
        ),
        summarize_validation_results(
            "Optimized CNN",
            champion_model,
            best_learning_rate,
            best_fc_size,
            champion_best_epoch,
            champion_val_results,
        ),
    ]
)

display(
    comparison_df.style.format(
        {
            "Learning rate": "{:.6f}",
            "Validation accuracy": "{:.2%}",
            "Macro precision": "{:.2%}",
            "Macro recall": "{:.2%}",
            "Macro F1": "{:.2%}",
        }
    )
)

## 9. Final Test Evaluation

The optimized model is now evaluated on the test set for the first and only time. Accuracy is reported together with a normalized confusion matrix and per-class precision, recall, and F1 scores so that class-specific weaknesses are visible.

In [ ]:
final_test_results = evaluate_model(champion_model, test_loader, champion_criterion)
final_test_accuracy = final_test_results["accuracy"]

test_precision, test_recall, test_f1, test_support = precision_recall_fscore_support(
    final_test_results["y_true"],
    final_test_results["y_pred"],
    labels=np.arange(n_classes),
    zero_division=0,
)

test_metrics_df = pd.DataFrame(
    {
        "Precision": test_precision,
        "Recall": test_recall,
        "F1": test_f1,
        "Support": test_support,
    },
    index=selected_classes,
)

print(f"Final test accuracy: {final_test_accuracy * 100:.2f}%")
display(test_metrics_df.style.format({"Precision": "{:.2%}", "Recall": "{:.2%}", "F1": "{:.2%}"}))

# Save the reproducible champion weights and replace the previous generated file.
model_path = Path("Champion_1D_CNN.pt")
torch.save(champion_model.state_dict(), model_path)
print(f"Optimized model weights saved to: {model_path}")

### 9.1 Comparison and Evaluation Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

validation_accuracies = comparison_df.set_index("Model")["Validation accuracy"] * 100
bars = axes[0, 0].bar(validation_accuracies.index, validation_accuracies.values, color=["#4C78A8", "#F58518"])
axes[0, 0].set_title("Validation Accuracy by Model")
axes[0, 0].set_ylabel("Accuracy (%)")
axes[0, 0].set_ylim(0, 100)
axes[0, 0].grid(axis="y", alpha=0.25)
for bar, value in zip(bars, validation_accuracies.values):
    axes[0, 0].text(bar.get_x() + bar.get_width() / 2, value + 1.5, f"{value:.1f}%", ha="center")

ConfusionMatrixDisplay.from_predictions(
    baseline_val_results["y_true"],
    baseline_val_results["y_pred"],
    display_labels=selected_classes,
    normalize="true",
    cmap="Blues",
    values_format=".2f",
    ax=axes[0, 1],
    colorbar=False,
)
axes[0, 1].set_title("Baseline Validation Confusion Matrix")

ConfusionMatrixDisplay.from_predictions(
    champion_val_results["y_true"],
    champion_val_results["y_pred"],
    display_labels=selected_classes,
    normalize="true",
    cmap="Oranges",
    values_format=".2f",
    ax=axes[0, 2],
    colorbar=False,
)
axes[0, 2].set_title("Optimized Validation Confusion Matrix")

ConfusionMatrixDisplay.from_predictions(
    final_test_results["y_true"],
    final_test_results["y_pred"],
    display_labels=selected_classes,
    normalize="true",
    cmap="Greens",
    values_format=".2f",
    ax=axes[1, 0],
    colorbar=False,
)
axes[1, 0].set_title("Final Test Confusion Matrix")

# Rotate the vertical class labels and separate them clearly from adjacent panels.
confusion_axes = [axes[0, 1], axes[0, 2], axes[1, 0]]
for confusion_ax in confusion_axes:
    confusion_ax.set_ylabel("True class", labelpad=18)
    confusion_ax.tick_params(axis="y", pad=14)
    plt.setp(
        confusion_ax.get_yticklabels(),
        rotation=90,
        horizontalalignment="center",
        verticalalignment="center",
    )

metric_positions = np.arange(n_classes)
metric_width = 0.25
axes[1, 1].bar(metric_positions - metric_width, test_precision * 100, width=metric_width, label="Precision", color="#4C78A8")
axes[1, 1].bar(metric_positions, test_recall * 100, width=metric_width, label="Recall", color="#F58518")
axes[1, 1].bar(metric_positions + metric_width, test_f1 * 100, width=metric_width, label="F1", color="#54A24B")
axes[1, 1].set_xticks(metric_positions, selected_classes)
axes[1, 1].set_ylim(0, 100)
axes[1, 1].set_title("Final Test Metrics by Class")
axes[1, 1].set_ylabel("Score (%)")
axes[1, 1].grid(axis="y", alpha=0.25)
axes[1, 1].legend()

axes[1, 2].axis("off")
axes[1, 2].text(
    0.05,
    0.85,
    "Final Result",
    fontsize=18,
    fontweight="bold",
    transform=axes[1, 2].transAxes,
)
axes[1, 2].text(
    0.05,
    0.63,
    f"Test accuracy: {final_test_accuracy * 100:.2f}%\n"
    f"Required threshold: 55.00%\n"
    f"Threshold met: {'Yes' if final_test_accuracy >= 0.55 else 'No'}\n\n"
    f"Best Optuna trial: {study.best_trial.number}\n"
    f"Learning rate: {best_learning_rate:.6f}\n"
    f"FC size: {best_fc_size}\n"
    f"Best training epoch: {champion_best_epoch}",
    fontsize=13,
    linespacing=1.5,
    va="top",
    transform=axes[1, 2].transAxes,
)

# Add generous whitespace and subtle separators so each panel reads as a unit.
fig.subplots_adjust(
    left=0.06,
    right=0.98,
    bottom=0.07,
    top=0.94,
    wspace=0.48,
    hspace=0.42,
)
fig.canvas.draw()

separator_color = "#D9D9D9"
for column in range(2):
    separator_x = (
        axes[0, column].get_position().x1
        + axes[0, column + 1].get_position().x0
    ) / 2
    fig.add_artist(
        Line2D(
            [separator_x, separator_x],
            [0.05, 0.96],
            transform=fig.transFigure,
            color=separator_color,
            linewidth=1,
            zorder=0,
        )
    )

separator_y = (
    axes[0, 0].get_position().y0
    + axes[1, 0].get_position().y1
) / 2
fig.add_artist(
    Line2D(
        [0.04, 0.99],
        [separator_y, separator_y],
        transform=fig.transFigure,
        color=separator_color,
        linewidth=1,
        zorder=0,
    )
)

plt.show()

## 10. Conclusion

In [ ]:
validation_improvement = (
    champion_val_results["accuracy"] - baseline_val_results["accuracy"]
) * 100

display(
    Markdown(
        f"""
The baseline 1D-CNN achieved **{baseline_val_results['accuracy'] * 100:.2f}% validation accuracy**. Hyperparameter optimization selected a learning rate of **{best_learning_rate:.6f}** and a fully connected size of **{best_fc_size}**. With these settings, the optimized model achieved **{champion_val_results['accuracy'] * 100:.2f}% validation accuracy**, a change of **{validation_improvement:+.2f} percentage points** relative to the baseline.

The optimized model reached **{final_test_accuracy * 100:.2f}% accuracy on the previously unseen test set**. Therefore, the required threshold of 55% was **{'met' if final_test_accuracy >= 0.55 else 'not met'}**. The confusion matrices and per-class metrics above provide the class-specific evidence needed to interpret this overall score.
"""
    )
)